# 02 - Baselines + MLflow + Fairlearn

Notebook na ordem recomendada para Fase 1:

1. Setup e carregamento
2. Tratamento minimo e split
3. Treino de baselines (DummyClassifier e Regressao Logistica)
4. Registro de experimentos no MLflow
5. Avaliacao de fairness com Fairlearn
6. Mitigacao de fairness e comparacao de trade-off

## 1) Setup e bibliotecas

Esta célula importa as bibliotecas usadas no experimento:
- manipulacao de dados (`pandas`, `numpy`)
- modelagem (`scikit-learn`)
- rastreabilidade (`mlflow`)
- fairness (`fairlearn`)

Resultado esperado: ambiente pronto para carregar dados, treinar baselines e medir desempenho/fairness.

In [2]:
import hashlib
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    equalized_odds_difference,
    false_positive_rate,
    selection_rate,
    true_positive_rate,
    true_negative_rate,
    false_negative_rate,
)
from fairlearn.reductions import EqualizedOdds, ExponentiatedGradient
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 200)

## 2) Carregamento do dataset

Esta célula lê o arquivo base de churn a partir de `../data/raw/telecom_churn_base_extended.csv`.

Resultado esperado:
- imprimir o shape bruto da base
- visualizar as primeiras linhas para validar colunas e tipos.

In [3]:
DATA_PATH = Path("../data/raw/telecom_churn_base_extended.csv")
df = pd.read_csv(DATA_PATH)
print("shape bruto:", df.shape)
display(df.head())

shape bruto: (51500, 73)


,customer_id,age,gender,region,plan_type,plan_price,is_promotional_plan,discount_amount,price_increase_last_3m,months_to_contract_end,contract_renewal_date,has_loyalty,months_to_loyalty_end,loyalty_end_date,internet_service,has_tv,has_fixed,tenure_months,network_outages_30d,avg_signal_quality,call_drop_rate,avg_internet_speed,service_failures_30d,installation_delay_days,repair_visits_90d,minutes_monthly,data_gb_monthly,sms_monthly,usage_delta_pct,days_since_last_usage,active_days_30d,night_usage_pct,roaming_usage,topup_frequency,avg_usage_last_3m,usage_trend_3m,payment_method,billing_type,late_payments_6m,invoice_shock_flag,avg_bill_last_6m,bill_variation_pct,collection_attempts_30d,monthly_charges,default_flag,days_past_due,support_calls_30d,support_calls_90d,complaints_30d,tickets_open_30d,last_complaint_reason,resolution_time_avg,first_call_resolution_flag,transferred_calls_count,last_contact_channel,app_login_30d,self_service_usage_30d,marketing_open_rate,campaign_response_flag,last_offer_accepted,days_since_last_interaction,portability_request_flag,sim_swap_count,competitor_offer_contact_flag,retention_offer_made,retention_offer_accepted,nps_score,nps_category,nps_promoter_flag,nps_detractor_flag,csat_score,churn,churn_probability
0,C000001,23,female,Sul,controle,38.73,0,0.00,0,12,2027-03-20,0,0,NaN,fiber,0,1,72,0,2.83,0.0025,32.65,0,0,0,349,0.12,315,0.379,57,3,39.7,0,1,0.12,0.251,boleto,digital,2,0,44.95,0.062,0,40.90,1,92,4,7,0,0,atraso_instalacao,0.72,0,0,telefone,54,3,39.0,0,0,102,0,0,0,0,0,10.0,promoter,1,0,5.0,1,0.9089
1,C000002,65,NaN,Nordeste,pos,55.93,1,16.97,0,28,2028-07-20,1,7,2026-10-20,none,0,1,92,0,4.30,0.0003,4.79,0,0,0,1998,1.13,17,-0.155,23,13,59.8,0,0,1.24,-0.157,debito,digital,0,0,39.76,-0.024,0,42.26,0,0,4,6,1,0,nenhum,1.08,0,0,telefone,43,11,45.8,1,0,52,0,1,0,0,0,6.0,detractor,0,1,4.0,1,0.2769
2,C000003,58,female,Sul,pos,61.65,0,0.00,0,0,2026-03-20,0,0,NaN,dsl,1,1,10,0,5.00,0.0029,0.30,0,5,0,521,10.46,267,-0.120,7,23,9.4,0,2,10.48,-0.342,card,digital,0,0,86.40,0.034,0,82.96,0,0,0,1,0,0,nenhum,0.92,1,0,app,31,0,51.2,1,0,50,0,0,0,0,0,10.0,promoter,1,0,5.0,0,0.0241
3,C000004,45,male,Sul,pos,63.73,0,0.00,0,22,2028-01-20,0,0,NaN,fiber,0,0,86,0,4.38,0.0016,12.29,0,0,1,2507,3.45,36,-0.388,24,26,20.6,0,1,4.13,-0.544,card,auto,0,0,79.99,-0.140,0,74.28,0,0,0,5,0,0,nenhum,0.12,0,0,app,13,26,23.4,0,0,173,0,0,0,0,0,10.0,promoter,1,0,5.0,0,0.0100
4,C000005,44,female,Sul,pos,64.58,0,0.00,1,30,2028-09-20,0,0,NaN,dsl,1,1,61,0,3.50,0.0014,25.65,1,0,0,1604,2.18,202,-0.223,44,10,5.0,0,2,2.79,-0.119,debito,paper,0,1,79.49,0.015,0,84.61,0,0,4,4,0,1,nenhum,0.43,0,0,telefone,38,27,44.7,0,0,21,0,0,0,0,0,7.0,passive,0,0,4.0,0,0.8085


## 3) Tratamento minimo dos dados

Esta etapa aplica limpeza simples para o baseline:
- remove duplicatas (incluindo por `customer_id`, quando existir)
- padroniza `gender`
- cria `age_group` a partir de `age`

Resultado esperado: base consistente para treino inicial, sem engenharia complexa.

In [4]:
# Tratamento minimo para baseline
df = df.drop_duplicates().copy()
if "customer_id" in df.columns:
    df = df.drop_duplicates(subset=["customer_id"], keep="first")

if "gender" in df.columns:
    df["gender"] = (
        df["gender"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({"": np.nan, "nan": np.nan})
        .fillna("unknown")
    )

if "age" in df.columns:
    bins = [0, 24, 34, 44, 54, 120]
    labels = ["<25", "25-34", "35-44", "45-54", "55+"]
    df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, include_lowest=True)
    df["age_group"] = df["age_group"].astype(str).replace({"nan": "unknown"})

print("shape tratado:", df.shape)
display(df[["gender", "age_group", "churn"]].head())

shape tratado: (49049, 74)


,gender,age_group,churn
0,female,<25,1
1,unknown,55+,1
2,female,55+,0
3,male,45-54,0
4,female,35-44,0


## 4) Definicao de alvo, split e preprocessamento

Aqui definimos:
- target (`churn`) e remocao de colunas com risco de leakage
- atributos sensiveis (`gender` e `age_group`) para analise de fairness
- split estratificado treino/teste (80/20)
- pipeline de preprocessamento (imputacao + one-hot)

Resultado esperado: `X_train`, `X_test`, `y_train`, `y_test` e `preprocess` prontos para treino.

In [5]:
target_col = "churn"
leakage_cols = ["customer_id", "churn_probability", "retention_offer_made", "retention_offer_accepted"]
drop_cols = [c for c in leakage_cols if c in df.columns]

X = df.drop(columns=[target_col] + drop_cols, errors="ignore")
y = df[target_col].astype(int)

sensitive_gender = df["gender"].astype(str) if "gender" in df.columns else pd.Series(["unknown"] * len(df), index=df.index)
sensitive_age = df["age_group"].astype(str) if "age_group" in df.columns else pd.Series(["unknown"] * len(df), index=df.index)

(
    X_train, X_test, y_train, y_test, s_gender_train, s_gender_test, s_age_train, s_age_test,
 ) = train_test_split(
    X, y, sensitive_gender, sensitive_age,
    test_size=0.2,
    stratify=y,
    random_state=42,
 )

num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[("imp", SimpleImputer(strategy="median"))]), num_cols),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ]
)

print("Treino/Teste:", X_train.shape, X_test.shape)

Treino/Teste: (39239, 69) (9810, 69)


## 5) Treino dos baselines e logging no MLflow

Esta etapa:
- define funcoes utilitarias (versao do dataset e ROC-AUC seguro)
- treina dois baselines (`dummy_stratified` e `log_reg`)
- calcula metricas de classificacao
- registra parametros e metricas no experimento `churn-baselines` no MLflow

Resultado esperado: tabela `perf_df` com comparacao de desempenho e identificacao do melhor baseline.

In [6]:
def compute_dataset_version(path: Path) -> str:
    data_bytes = path.read_bytes()
    digest = hashlib.sha256(data_bytes).hexdigest()[:12]
    return f"sha256:{digest}"


def safe_roc_auc(y_true, y_score) -> float:
    # Evita quebra se y_score tiver variancia zero em algum baseline.
    try:
        return float(roc_auc_score(y_true, y_score))
    except ValueError:
        return float("nan")


# Parametros mais robustos para OHE de alta dimensionalidade (reduz chance de nao convergencia).
LOG_REG_KWARGS = {
    "max_iter": 2000,
    "solver": "liblinear",
    "random_state": 42,
}

dataset_version = compute_dataset_version(DATA_PATH)
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("churn-baselines")

models = {
    "dummy_stratified": DummyClassifier(strategy="stratified", random_state=42),
    "log_reg": LogisticRegression(**LOG_REG_KWARGS),
}

trained = {}
pred_store = {}
rows = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[("prep", preprocess), ("model", estimator)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test).astype(int)
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_test)[:, 1]
    else:
        y_score = y_pred.astype(float)

    metrics_row = {
        "model": name,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred)),
        "roc_auc": safe_roc_auc(y_test, y_score),
        "pr_auc": float(average_precision_score(y_test, y_score)),
        "positive_rate": float(y_pred.mean()),
    }
    rows.append(metrics_row)

    with mlflow.start_run(run_name=name):
        mlflow.log_params(
            {
                "model": name,
                "test_size": 0.2,
                "random_state": 42,
                "dataset_path": str(DATA_PATH),
                "dataset_version": dataset_version,
                "n_rows": int(len(df)),
                "n_features_before_ohe": int(X.shape[1]),
            }
        )
        mlflow.log_metrics({k: v for k, v in metrics_row.items() if k != "model" and pd.notna(v)})

    trained[name] = pipe
    pred_store[name] = {"y_pred": y_pred, "y_score": y_score}

perf_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False, na_position="last").reset_index(drop=True)
display(perf_df)
best_name = perf_df.loc[0, "model"]
best_model = trained[best_name]
print("Melhor baseline por ROC-AUC:", best_name)
print("Dataset version:", dataset_version)

,model,accuracy,f1,roc_auc,pr_auc,positive_rate
0,log_reg,0.804893,0.739875,0.878567,0.848536,0.356371
1,dummy_stratified,0.529358,0.393697,0.504595,0.395905,0.382569


Melhor baseline por ROC-AUC: log_reg
Dataset version: sha256:3a1b88dbc248


## 6) Avaliacao de fairness por grupo sensivel

Com as predicoes dos baselines, esta célula calcula indicadores de fairness por:
- `gender`
- `age_group`

Metricas principais:
- `dp_diff` (demographic parity)
- `eo_diff` (equalized odds)
- gaps de taxa de selecao, TPR e FPR

Resultado esperado: `fairness_df` comparando disparidades entre grupos para cada modelo.

In [7]:
fair_rows = []

for model_name, payload in pred_store.items():
    y_pred_model = payload["y_pred"]

    for sensitive_name, sensitive_values in [
        ("gender", s_gender_test),
        ("age_group", s_age_test),
    ]:
        mf = MetricFrame(
            metrics={
                "selection_rate": selection_rate,
                "tpr": true_positive_rate,
                "tnr": true_negative_rate,
                "fpr": false_positive_rate,
                "fnr": false_negative_rate,
            },
            y_true=y_test,
            y_pred=y_pred_model,
            sensitive_features=sensitive_values,
        )

        fair_rows.append(
            {
                "model": model_name,
                "sensitive_feature": sensitive_name,
                "dp_diff": float(
                    demographic_parity_difference(
                        y_test, y_pred_model, sensitive_features=sensitive_values
                    )
                ),
                "eo_diff": float(
                    equalized_odds_difference(
                        y_test, y_pred_model, sensitive_features=sensitive_values
                    )
                ),
                "selection_rate_gap": float(mf.difference(method="between_groups")["selection_rate"]),
                "tpr_gap": float(mf.difference(method="between_groups")["tpr"]),
                "fpr_gap": float(mf.difference(method="between_groups")["fpr"]),
            }
        )

fairness_df = pd.DataFrame(fair_rows).sort_values(["model", "sensitive_feature"]).reset_index(drop=True)
display(fairness_df)

,model,sensitive_feature,dp_diff,eo_diff,selection_rate_gap,tpr_gap,fpr_gap
0,dummy_stratified,age_group,0.031805,0.067957,0.031805,0.048495,0.067957
1,dummy_stratified,gender,0.023419,0.036503,0.023419,0.036503,0.025367
2,log_reg,age_group,0.034179,0.135786,0.034179,0.135786,0.039362
3,log_reg,gender,0.044792,0.066339,0.044792,0.052598,0.066339


## 7) Mitigacao de fairness com Equalized Odds

Aqui aplicamos mitigacao no baseline logistico usando `ExponentiatedGradient` com restricao `EqualizedOdds`.

Decisoes de implementacao:
- usa uma amostra estratificada de treino para reduzir tempo de execucao
- mede tempo de treinamento da mitigacao
- compara metricas de qualidade e fairness no conjunto de teste
- registra o experimento mitigado no MLflow

Resultado esperado: tabela com metricas do modelo mitigado para comparar trade-off entre performance e equidade.

In [ ]:
# Mitigacao de fairness no modelo logistico com EqualizedOdds (versao mais rapida)
from time import perf_counter

# Mitigacao em amostra estratificada para reduzir tempo de execucao.
MITIGATION_SAMPLE_SIZE = 15000
if len(X_train) > MITIGATION_SAMPLE_SIZE:
    X_mit, _, y_mit, _, s_gender_mit, _ = train_test_split(
        X_train,
        y_train,
        s_gender_train,
        train_size=MITIGATION_SAMPLE_SIZE,
        stratify=y_train,
        random_state=42,
    )
else:
    X_mit, y_mit, s_gender_mit = X_train, y_train, s_gender_train

X_mit_t = preprocess.fit_transform(X_mit)
X_test_t = preprocess.transform(X_test)

mitigation_estimator = LogisticRegression(
    max_iter=400,
    solver="liblinear",
    random_state=42,
)

mitigator = ExponentiatedGradient(
    estimator=mitigation_estimator,
    constraints=EqualizedOdds(),
    eps=0.02,
    max_iter=15,
)

t0 = perf_counter()
mitigator.fit(X_mit_t, y_mit, sensitive_features=s_gender_mit)
elapsed = perf_counter() - t0
print(f"Tempo de mitigacao: {elapsed:.1f}s (n_treino_mitig={len(X_mit):,})")

y_pred_mitig = mitigator.predict(X_test_t).astype(int)

mitig_metrics = {
    "model": "log_reg_mitigated_equalized_odds",
    "accuracy": float(accuracy_score(y_test, y_pred_mitig)),
    "f1": float(f1_score(y_test, y_pred_mitig)),
    "positive_rate": float(y_pred_mitig.mean()),
    "dp_diff_gender": float(
        demographic_parity_difference(y_test, y_pred_mitig, sensitive_features=s_gender_test)
    ),
    "eo_diff_gender": float(
        equalized_odds_difference(y_test, y_pred_mitig, sensitive_features=s_gender_test)
    ),
}

with mlflow.start_run(run_name="log_reg_mitigated_equalized_odds"):
    mlflow.log_params(
        {
            "model": "log_reg_mitigated_equalized_odds",
            "base_model": "log_reg",
            "constraint": "EqualizedOdds",
            "sensitive_feature": "gender",
            "dataset_path": str(DATA_PATH),
            "dataset_version": dataset_version,
            "mitigation_sample_size": int(len(X_mit)),
            "mitigation_eps": 0.02,
            "mitigation_max_iter": 15,
        }
    )
    mlflow.log_metrics({k: v for k, v in mitig_metrics.items() if k != "model" and pd.notna(v)})

display(pd.DataFrame([mitig_metrics]))

Tempo de mitigacao: 50.8s (n_treino_mitig=15,000)


,model,accuracy,f1,positive_rate,dp_diff_gender,eo_diff_gender
0,log_reg_mitigated_equalized_odds,0.805607,0.738946,0.350968,0.056138,0.073859


## Leitura dos resultados

- Compare DummyClassifier vs Regressao Logistica em accuracy, F1, ROC-AUC e PR-AUC.
- Consulte o experimento `churn-baselines` no MLflow para parametros, metricas e versao do dataset.
- Revise os gaps de fairness (DP/EO, TPR/FPR gap) por `gender` e `age_group`.
- Compare baseline logistico com versao mitigada por EqualizedOdds antes de decidir o modelo operacional.